# Phase 2 & 3: Sen1Floods11 Dataset Exploration & Verification
### AI-Based Flood Detection and Mapping Using Satellite Imagery

This notebook provides a complete interactive workflow to:
1. Verify the environment and dataset files.
2. Inspect multi-modal satellite layers (Sentinel-1 SAR, Sentinel-2 Optical, JRC baseline).
3. Analyze label distributions, class imbalance, and no-data masks.
4. Visualize multi-panel chip comparisons and flood overlays.
5. Verify PyTorch Dataset & DataLoader batch generation.

In [ ]:
# Step 1: Import core dependencies
import os
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.data.validation import run_dataset_validation, discover_chips
from src.data.preprocessing import FloodPreprocessor, SynchronizedAugmentation
from src.data.dataset import Sen1Floods11Dataset, create_dataloaders
from src.visualization.visualize_data import plot_single_sample

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon) Available: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

In [ ]:
# Step 2: Configure paths and run dataset validation
data_dir = str(project_root / "data" / "samples")
splits_dir = str(project_root / "data" / "raw" / "splits")
output_dir = str(project_root / "outputs")

summary = run_dataset_validation(data_dir=data_dir, output_dir=output_dir)
print(json.dumps(summary, indent=2))

In [ ]:
# Step 3: Inspect dataset statistics dataframe
stats_df = pd.read_csv(project_root / "outputs" / "dataset_statistics.csv")
print(f"Total Chips: {len(stats_df)}")
stats_df.head(10)

In [ ]:
# Step 4: Class imbalance and water coverage distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution pie chart
pixel_dist = summary["pixel_class_distribution"]
labels = ['Non-Water Land', 'Inundated Water', 'Invalid / Cloud']
sizes = [pixel_dist['non_water_pixels'], pixel_dist['water_pixels'], pixel_dist['invalid_cloud_nodata_pixels']]
colors = ['#4a5568', '#3182ce', '#ecc94b']

ax[0].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0, 0.08, 0.05))
ax[0].set_title('Global Pixel Distribution across Dataset Chips', fontweight='bold')

# Per-event flood percentage histogram
event_coverage = stats_df.groupby('event')['flood_water_ratio'].mean() * 100
event_coverage.plot(kind='bar', color='#3182ce', ax=ax[1])
ax[1].set_title('Average Flood Water Coverage by Geographic Event (%)', fontweight='bold')
ax[1].set_ylabel('Flood Coverage (%)')
ax[1].set_xlabel('Event / Country')
plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Visualize a multi-panel chip sample with flood overlay
chips = discover_chips(data_dir)
sample_stem = list(chips.keys())[0]
sample_fig_path = str(project_root / "outputs" / "visualizations" / "dataset_samples" / f"{sample_stem}_verification.png")

plot_single_sample(sample_stem, data_dir, sample_fig_path)

# Display the generated figure
from IPython.display import Image
Image(sample_fig_path)

In [ ]:
# Step 6: Test PyTorch Dataset and Multi-Modal Inputs
modalities = ['s1', 's2_rgb', 's1_jrc', 's1_s2_rgb', 's1_s2_rgb_jrc']
for mod in modalities:
    ds = Sen1Floods11Dataset(data_dir=data_dir, modality=mod)
    sample = ds[0]
    print(f"Modality: {mod:15s} | Channels: {ds.in_channels} | Image Shape: {sample['image'].shape} | Mask Shape: {sample['mask'].shape}")

In [ ]:
# Step 7: Test PyTorch DataLoader Batch Iteration
loaders = create_dataloaders(data_dir=data_dir, splits_dir=splits_dir, modality='s1', batch_size=4)
train_batch = next(iter(loaders['train']))

print("=== Train Batch Properties ===")
print(f"Images Tensor: {train_batch['image'].shape} (dtype: {train_batch['image'].dtype}, range: [{train_batch['image'].min():.2f}, {train_batch['image'].max():.2f}])")
print(f"Masks Tensor:  {train_batch['mask'].shape} (dtype: {train_batch['mask'].dtype}, unique values: {torch.unique(train_batch['mask']).tolist()})")